![logo_itq](img/logo-itq.jpeg)
## HoldOut + Comparación de Modelos (Heart Disease)
*Nixon Malquin* — 24/05/2026

Pipeline final del proyecto:
1. Cargar dataset limpio (`heart_clean.csv`) generado en Programa 4.
2. Quedarnos con los 8 atributos seleccionados en Programa 6.
3. Split externo 80/20 + estandarización + split interno 80/20.
4. Comparar varios modelos.
5. Guardar el mejor en `models/`.

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
# Dataset limpio (sin duplicados)
df = pd.read_csv('../dataset/heart_clean.csv')
print('Tabla de datos: %d instancias y %d atributos' % (df.shape[0], df.shape[1]-1))
print('Valores de la clase:', set(df['target']))
valores, ocur = np.unique(df['target'], return_counts=True)
print(valores, ocur)

In [ ]:
# Atributos seleccionados en el Programa 6
SELECTED = ['sex', 'cp', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
X = df[SELECTED].values
y = df['target'].values
print('Atributos usados:', SELECTED)

In [ ]:
# Test: hold-out 80/20 estratificado (partición externa)
X_training, X_test, y_training, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
valores_test, ocur_test = np.unique(y_test, return_counts=True)
print('Test:', 'clases:', valores_test, ' ocurrencias:', ocur_test)

In [ ]:
# Estandarización (fit solo sobre training)
standardizer = StandardScaler()
X_training = standardizer.fit_transform(X_training)
X_test     = standardizer.transform(X_test)

In [ ]:
# Validación: hold-out 80/20 estratificado (partición interna)
X_train, X_val, y_train, y_val = train_test_split(
    X_training, y_training, test_size=0.2, random_state=42, stratify=y_training)
print('Entrenamiento:', np.unique(y_train, return_counts=True))
print('Validation:   ', np.unique(y_val, return_counts=True))

In [ ]:
# Baseline: DummyClassifier
clf = DummyClassifier(strategy='prior', random_state=42)
clf.fit(X_train, y_train)
print('Dummy val :', round(clf.score(X_val, y_val)*100, 2), '%')
print('Dummy test:', round(clf.score(X_test, y_test)*100, 2), '%')

## Comparación de modelos
Probamos varios algoritmos para escoger el más preciso.

In [ ]:
modelos = {
    'SVC (C=0.5, rbf)':      SVC(C=0.5),
    'SVC (C=10, rbf)':       SVC(C=10),
    'Logistic Regression':   LogisticRegression(max_iter=2000),
    'Decision Tree':         DecisionTreeClassifier(random_state=42),
    'Random Forest (100)':   RandomForestClassifier(n_estimators=100, random_state=42),
    'Random Forest (300)':   RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(random_state=42),
    'KNN (k=5)':             KNeighborsClassifier(n_neighbors=5),
    'MLP (50,25)':           MLPClassifier(hidden_layer_sizes=(50,25), max_iter=2000, random_state=42),
}

print(f"{'Modelo':<25} {'val_acc':>10} {'test_acc':>10}")
print('-'*50)
resultados = []
for n, m in modelos.items():
    m.fit(X_train, y_train)
    va = m.score(X_val, y_val); te = m.score(X_test, y_test)
    resultados.append((n, va, te, m))
    print(f'{n:<25} {va*100:>9.2f}% {te*100:>9.2f}%')

print('\n>>> Top 3 por test_acc:')
for n,va,te,_ in sorted(resultados, key=lambda x: -x[2])[:3]:
    print(f'    {n:<25} test={te*100:.2f}%')

In [ ]:
# El ganador es Random Forest. Lo entrenamos sobre TODO el training disponible y guardamos.
mejor = RandomForestClassifier(n_estimators=300, random_state=42)
mejor.fit(X_training, y_training)  # usamos training completo
print('Exactitud final en test:', round(mejor.score(X_test, y_test)*100, 2), '%')

with open('../models/model.pickle', 'wb') as fw:
    pickle.dump(mejor, fw)
with open('../models/scaler.pkl', 'wb') as fw:
    pickle.dump(standardizer, fw)
with open('../models/features.pkl', 'wb') as fw:
    pickle.dump(SELECTED, fw)
print('Modelo, scaler y atributos guardados en ../models/')

In [ ]:
# Probar el modelo cargado con datos ingresados por teclado
with open('../models/model.pickle', 'rb') as f: modelo = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f: scaler_cargado = pickle.load(f)

sex     = float(input('Sexo 0=mujer 1=hombre: '))
cp      = float(input('Tipo dolor pecho (0-3): '))
thalach = float(input('Frecuencia cardiaca maxima: '))
exang   = float(input('Angina inducida por ejercicio (0/1): '))
oldpeak = float(input('Depresion ST: '))
slope   = float(input('Pendiente segmento ST (0-2): '))
ca      = float(input('Vasos coloreados (0-4): '))
thal    = float(input('Talasemia (0-3): '))

In [ ]:
nuevo_dato = [[sex, cp, thalach, exang, oldpeak, slope, ca, thal]]
nuevo_dato_esc = scaler_cargado.transform(nuevo_dato)
prediccion = modelo.predict(nuevo_dato_esc)
# En este dataset target=0 = CON enfermedad, target=1 = SIN enfermedad
if prediccion == 0:
    print('El paciente SI presenta indicios de enfermedad cardiaca', prediccion)
else:
    print('El paciente NO presenta indicios de enfermedad cardiaca', prediccion)

### Link de repositorio
https://github.com/Ngmalquin123/MachingITQ